# Load the minimal LiveWell Resident 360 (Fabric)

Runs **inside Fabric** with `lh_resident360` attached as the default lakehouse. `scripts/fabric/deploy.sh` imports it
and runs it for you; you can also press **Run all**.

It reads the vendored kit files that `scripts/fabric/20-lakehouse-load.sh` uploaded to `Files/livewell/`, rebuilds the
gold layer with the **same code** as the local mirror (`scripts/r360.py`, uploaded next to the data), upserts Rahim
(`RESIDENT_00061`, SPEC.md §3.3) from `rahim.json`, and writes six Delta tables:

| Table | Grain | Ontology use |
|---|---|---|
| `resident_360` | one row per resident | Resident entity; `livesIn` |
| `dim_region` | one row per region | Region entity |
| `dim_programme` | one row per programme | Programme entity |
| `dim_event_occurrence` | one row per event occurrence | EventOccurrence entity; `heldIn` |
| `fact_event_attendance` | attended resident × occurrence pairs only | `attended` |
| `fact_programme_enrolment` | every enrolment row | `enrolledIn` |

Synthetic data only. Nothing here identifies a real person.

In [ ]:
# Parameters (deploy.sh may override them when it runs the notebook as a job)
FILES_ROOT = "/lakehouse/default/Files/livewell"   # uploaded by 20-lakehouse-load.sh
EXPORT_DIR = "/lakehouse/default/Files/export"     # load_summary.json + resident_360.csv for local scripts
TABLE_SCHEMA = ""                                  # lh_resident360 has no schemas; "dbo" for a schema-enabled lakehouse

## 1 · Rebuild the gold layer with the shared logic

In [ ]:
import sys, json, datetime as dt

sys.dont_write_bytecode = True               # no __pycache__ in OneLake
from pathlib import Path

sys.path.insert(0, f"{FILES_ROOT}/scripts")
import r360                                  # same module the local scripts use

r360.DATA = Path(f"{FILES_ROOT}/resident360")  # read the uploaded kit files instead of the repo copy
tables = r360.build()
for name, rows in tables.items():
    print(f"{name:26s} {len(rows):>6d} rows")

## 2 · Check Rahim before writing anything

The notebook stops here if Rahim's row drifts from the persona pinned in `rahim.json` (`expected`).

In [ ]:
rahim = r360.load_rahim()
facts = r360.rahim_facts(tables)
exp = rahim["expected"]
problems = []
for key, want in exp.items():
    if key.endswith("_max"):
        field = key[:-4]
        if facts.get(field) is None or facts[field] > want:
            problems.append(f"{field}={facts.get(field)} (want <= {want})")
    elif facts.get(key) != want:
        problems.append(f"{key}={facts.get(key)!r} (want {want!r})")
print(json.dumps(facts, indent=2))
if problems:
    raise AssertionError("Rahim does not match rahim.json: " + "; ".join(problems))
print("Rahim OK:", rahim["resident_id"], "|", facts["region"], "|", facts["age_band"], "| disengaged =", facts["is_disengaged"])

## 3 · Write the Delta tables

Column types come from the values: whole numbers become `long` (ontology `BigInt`), decimals `double`, text `string`.
Date columns (`*_date`, `first_enrolment`) become `timestamp` so they bind to ontology `DateTime` properties.

In [ ]:
from pyspark.sql import types as T

DATE_COLUMNS = {"first_enrolment"}

def is_date_col(col):
    return col.endswith("_date") or col in DATE_COLUMNS

def to_frame(rows):
    cols = list(rows[0].keys())
    fields, casts = [], {}
    for c in cols:
        vals = [r[c] for r in rows if r[c] is not None]
        if is_date_col(c):
            fields.append(T.StructField(c, T.TimestampType(), True)); casts[c] = "ts"
        elif vals and all(isinstance(v, bool) for v in vals):
            fields.append(T.StructField(c, T.BooleanType(), True))
        elif vals and all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            fields.append(T.StructField(c, T.LongType(), True))
        elif vals and all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
            fields.append(T.StructField(c, T.DoubleType(), True)); casts[c] = "float"
        else:
            fields.append(T.StructField(c, T.StringType(), True))
    def conv(r):
        out = []
        for c in cols:
            v = r[c]
            if v is not None and casts.get(c) == "ts":
                v = dt.datetime.fromisoformat(str(v)[:10])
            elif v is not None and casts.get(c) == "float":
                v = float(v)
            out.append(v)
        return tuple(out)
    return spark.createDataFrame([conv(r) for r in rows], T.StructType(fields))

prefix = f"{TABLE_SCHEMA}." if TABLE_SCHEMA else ""
counts = {}
for name, rows in tables.items():
    df = to_frame(rows)
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(prefix + name))
    counts[name] = spark.table(prefix + name).count()
    print(f"wrote {prefix + name:34s} {counts[name]:>6d} rows")

mismatch = {n: (counts[n], len(tables[n])) for n in tables if counts[n] != len(tables[n])}
if mismatch:
    raise AssertionError(f"row counts differ after write: {mismatch}")

## 4 · Export the summary for the local scripts

`scripts/fabric/deploy.sh` downloads `Files/export/load_summary.json` to confirm the load, and
`scripts/gen-citizens.py --from-onelake` reads `Files/export/resident_360.csv` and `fact_programme_enrolment.csv`.

In [ ]:
import csv, os
os.makedirs(EXPORT_DIR, exist_ok=True)
for name in ("resident_360", "fact_programme_enrolment"):      # read by gen-citizens.py --from-onelake
    with open(f"{EXPORT_DIR}/{name}.csv", "w", encoding="utf-8", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(tables[name][0].keys()), lineterminator="\n")
        w.writeheader(); w.writerows(tables[name])
summary = {
    "generated_at": dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
    "table_schema": TABLE_SCHEMA,
    "counts": counts,
    "rahim": facts,
    "reference_answers": r360.reference_answers(tables),
}
with open(f"{EXPORT_DIR}/load_summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(json.dumps(summary["counts"], indent=2))
print("top disengaged region:", summary["reference_answers"]["q_disengaged_regions"]["top_region"])